# Fatura & Fiş OCR — Kaggle Çalışma DefteriKod **VS Code'da yazılır**, GitHub'a push edilir, bu defter repo'yu klonlayıpçalıştırır. Burada kod yazılmaz.**Notebook ayarları** (sağ panel → Settings):| Ayar | Değer ||---|---|| Accelerator | GPU T4 x2 || Internet | On || Persistence | Variables and Files |**Veri seti:** sağ panel → Add Input → `SROIE` ara → *SROIE datasetv2* ekle.**Secret:** Add-ons → Secrets → `NGROK_AUTHTOKEN` (sadece servis bölümü için).

## 0 · GPU kontrolü`Tesla T4` ve ~15360MiB görmelisin.

In [ ]:
!nvidia-smi

## 1 · Repo'yu klonlaKodu her güncellediğinde bu hücreyi çalıştır. Eski kopyayı silip temiz çeker.**Dikkat:** bu hücre `/kaggle/working/invoice-ocr-service` klasörünü komplesiler. Adaptör ve sonuç dosyaları bilerek `/kaggle/working` altında tutuluyor,o yüzden onlar etkilenmiyor.

In [ ]:
import os, shutil
os.environ['GIT_TERMINAL_PROMPT'] = '0'
%cd /kaggle/working
if os.path.exists('/kaggle/working/invoice-ocr-service'):
    shutil.rmtree('/kaggle/working/invoice-ocr-service')
!git clone https://github.com/betulerkocc/invoice-ocr-service.git /kaggle/working/invoice-ocr-service
%cd /kaggle/working/invoice-ocr-service

## 2 · Oturum kurulumu — **her yeniden başlatmada çalıştır**Kaggle oturumu kapandığında (bağlantı koptu, sekme kapandı, süre doldu) deftertemel imaja döner: kurduğun paketler silinir, kaldırdıkların geri gelir,`%cd` unutulur. Bu hücre üçünü birden halleder.Beklenen son iki satır:```peft + bitsandbytes: tamamModuleNotFoundError: No module named 'torchao'```İkincisi hata gibi görünür ama **istediğimiz budur** — `torchao` kaldırılmışdemektir. Kaggle'daki sürümü `peft`'in beklediğinden eski ve `peft` uyumsuzsürümü görünce hata fırlatıp adaptörü yükletmiyor. Biz torchao kullanmıyoruz.

In [ ]:
%cd /kaggle/working/invoice-ocr-service
!python -m pip install -U "peft>=0.13" "bitsandbytes>=0.46.1" "accelerate>=0.34"
!python -m pip uninstall -y torchao
print("
--- kontrol ---")
!python -c "import peft, bitsandbytes; print('peft + bitsandbytes: tamam')"
!python -c "import torchao" 2>&1 | tail -1

## 3 · Veriyi hazırlaSROIE'yi eğitim formatına çevirir. Klonlama `data/` klasörünü sildiği içinher klonlamadan sonra tekrar çalıştırılmalı — saniyeler sürer.Beklenen: `train: 626`, `test: 347`. `invoice_no` oranının düşük olması(%35-45) **normaldir** — fişlerin çoğunda fatura numarası gerçekten yok,orada `null` doğru etikettir.

In [ ]:
!python llm/finetune/prepare_sroie.py

## 4 · Eğitim### 4a · Duman testi — önce bunu çalıştır20 örnekle ~10 saniye. Amaç doğruluk değil, boru hattının aktığını görmek.Buradaki bir hatayı 11 dakikanın sonunda keşfetmek istemezsin.`gorunur kart: 1` ve `trainable%: ~1.75` görmelisin.

In [ ]:
!python llm/finetune/train.py --limit 20 --epochs 1

### 4b · Gerçek eğitim626 örnek × 3 epoch, tek T4'te ~11 dakika.`--out-dir` bayrağı **önemli**: adaptörü repo klasörünün dışına yazar, böylecesonraki klonlama onu silemez.

In [ ]:
!python llm/finetune/train.py --out-dir /kaggle/working/qwen-invoice-lora

## 5 · Adaptörü yedekle`/kaggle/working` oturum kapanınca silinebilir ve **tek pahalı şey adaptör**.Zip'ledikten sonra sağ panel → Output → `qwen-invoice-lora.zip` → indir.İçinde `adapter_model.safetensors` olduğunu doğrula.

In [ ]:
!cd /kaggle/working && zip -r qwen-invoice-lora.zip qwen-invoice-lora

## 6 · DeğerlendirmeBase model ile fine-tune edilmiş modeli 347 test örneğinde karşılaştırır.**Staj raporunun asıl çıktısı bu.**Test örneklerini model eğitimde hiç görmedi — ezberi değil genellemeyi ölçüyoruz.### 6a · Hızlı bakış (50 örnek, ~6 dk)

In [ ]:
!python scripts/evaluate.py --compare /kaggle/working/qwen-invoice-lora --limit 50

### 6b · Tam tur (347 örnek, ~25 dk)Rapora yazılacak sayılar buradan. Sonuç dosyası repo dışına yazılıyor kiklonlama silmesin.

In [ ]:
!python scripts/evaluate.py --compare /kaggle/working/qwen-invoice-lora --out /kaggle/working/sonuclar.json

---# Servis (ngrok) — opsiyonelBuradan aşağısı sadece **gerçek bir fatura fotoğrafını Postman'den test etmek**istediğinde gerekli. Eğitim ve değerlendirme için çalıştırmana gerek yok.## 7 · Servis paketleriPaddle büyük bir paket, kurulum birkaç dakika sürer.

In [ ]:
!python -m pip install pyngrok "paddlepaddle==3.1.1" paddleocr

## 8 · ngrok token'ıToken'ı Add-ons → Secrets'tan okuyup ortam değişkenine koyar. `!python` ayrıbir süreç başlattığı ve Secret erişimi ona her zaman taşınmadığı için bu adımgerekli.Token'ın sadece ilk 6 karakteri basılır — defteri paylaşırsan tamamı görünmesin.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['NGROK_AUTHTOKEN'] = UserSecretsClient().get_secret('NGROK_AUTHTOKEN')
print("token okundu:", os.environ['NGROK_AUTHTOKEN'][:6] + "...")

## 9 · Servisi başlat`ADAPTER_PATH` fine-tune edilmiş modeli servise takar. Boş bırakırsan basemodel çalışır.Bu hücre **bilerek durmaz** — çalıştığı sürece servis ayakta. Durdurmak için ⏹.Çıktıdaki `PUBLIC URL` satırındaki **gerçek adresi** kopyala (her başlatmadadeğişir). `INFO: Uvicorn running` satırını görmeden istek atma.**Postman:** `POST <adres>/analyze-invoice` → Body → form-data → key `file`,tipini **File** yap → fatura fotoğrafını seç.

In [ ]:
import os
os.environ['ADAPTER_PATH'] = '/kaggle/working/qwen-invoice-lora'
!python scripts/serve_kaggle.py